# Google Sheet Integration
## Install libraries

In [1]:
get_ipython().system('pip install -q gspread google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client pandas')
print("If no red 'ERROR' text appeared above, the libraries installed correctly.")


If no red 'ERROR' text appeared above, the libraries installed correctly.


## Import libraries 

In [2]:
import os
import json
from datetime import datetime

import pandas as pd
import gspread
from google.oauth2.service_account import Credentials
from gspread.exceptions import SpreadsheetNotFound, WorksheetNotFound, APIError

print("All imports succeeded.")


All imports succeeded.


## Authentication

In [3]:
CREDENTIALS_PATH = "credentials.json"

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]

gc = None
service_account_email = None

if not os.path.exists(CREDENTIALS_PATH):
    print(f"Failed Authentication")
    print(f"Reason: '{CREDENTIALS_PATH}' was not found in the current working directory:")
    print(f"    {os.getcwd()}")
    print("Fix: upload/copy credentials.json into this exact folder (in Colab, re-upload it every")
    print("     session — the /content directory is wiped when the runtime restarts), then re-run this cell.")
else:
    try:
        with open(CREDENTIALS_PATH) as f:
            key_data = json.load(f)
        service_account_email = key_data.get("client_email")

        creds = Credentials.from_service_account_file(CREDENTIALS_PATH, scopes=SCOPES)
        gc = gspread.authorize(creds)

        print("Authentication Successful")
        print(f"Service account email: {service_account_email}")
        print("Double-check this exact email has Editor access on your target Google Sheet (Share button).")
    except json.JSONDecodeError as error:
        print(f"Failed Authentication")
        print(f"Reason: '{CREDENTIALS_PATH}' is not valid JSON: {error}")
        print("Fix: re-download the key from Google Cloud Console > IAM & Admin > Service Accounts > Keys.")
    except Exception as error:
        print(f"Failed Authentication")
        print(f"Reason (exact exception): {type(error).__name__}: {error}")
        print("Common causes: revoked/deleted key, disabled service account, malformed credentials.json.")


Authentication Successful
Service account email: lead-management-bot@lead-management-system-503309.iam.gserviceaccount.com
Double-check this exact email has Editor access on your target Google Sheet (Share button).


## Connect to the Google Sheet by Spreadsheet ID

In [4]:
SPREADSHEET_ID = "1R_TNrW-PXvOp8G5dZduDy1Oj6YDeWVzAc8aXyzUniiQ"
WORKSHEET_NAME = "Sheet1"   

spreadsheet = None
worksheet = None

if gc is None:
    print("Skipping: authentication in Cell 3 did not succeed.")
else:
    try:
        spreadsheet = gc.open_by_key(SPREADSHEET_ID)
        print("Connected to Spreadsheet")
        print(f"Spreadsheet title: {spreadsheet.title}")
    except SpreadsheetNotFound:
        print("Invalid Spreadsheet ID")
        print(f"Reason: no spreadsheet found for ID '{SPREADSHEET_ID}'.")
        print("Fix: confirm the ID is copied exactly from the sheet's URL (the string between /d/ and /edit),")
        print("     and that the service account's email has been given Editor access via Share.")
    except APIError as error:
        print("Permission Denied / API Error")
        print(f"Reason (exact exception): {error}")
        print("Fix: make sure the Google Sheets API AND Google Drive API are both enabled on this")
        print("     project in Google Cloud Console, and that the sheet is shared with the service account email.")
    except Exception as error:
        print("Unexpected error while opening spreadsheet")
        print(f"{type(error).__name__}: {error}")

    if spreadsheet is not None:
        all_titles = [ws.title for ws in spreadsheet.worksheets()]
        print(f"Worksheets found in this spreadsheet: {all_titles}")

        try:
            worksheet = spreadsheet.worksheet(WORKSHEET_NAME)
            print(f"Connected to Worksheet '{WORKSHEET_NAME}'")
        except WorksheetNotFound:
            print(f"Incorrect Worksheet Name: '{WORKSHEET_NAME}' does not exist in this spreadsheet.")
            worksheet = spreadsheet.get_worksheet(0)
            print(f"Falling back to the first tab instead: '{worksheet.title}'")
            print(f"If this is wrong, set WORKSHEET_NAME = '{worksheet.title}' above (or the correct tab name) and re-run.")

        if worksheet is not None:
            print(f"Total rows (incl. header): {worksheet.row_count}")
            print(f"Total columns: {worksheet.col_count}")


Connected to Spreadsheet
Spreadsheet title: client_email
Worksheets found in this spreadsheet: ['Sheet1']
Connected to Worksheet 'Sheet1'
Total rows (incl. header): 1000
Total columns: 26


## Verify worksheet headers

In [5]:
EXPECTED_COLUMNS = [
    "lead_id",
    "company_name",
    "contact_name",
    "email",
    "phone",
    "source",
    "status",
    "assigned_to",
    "priority",
    "country",
    "industry",
    "last_contact_date",
    "notes",
]

if worksheet is None:
    print("Skipping: no worksheet connected (see Cell 4).")
else:
    try:
        existing_headers = worksheet.row_values(1)
        print(f"Current headers found in row 1: {existing_headers}")

        if existing_headers == EXPECTED_COLUMNS:
            print("Headers Verified — they match exactly.")
        else:
            print("Header mismatch detected. Fixing automatically...")
            worksheet.clear()
            worksheet.insert_row(EXPECTED_COLUMNS, index=1)
            print(f"Headers rewritten to: {EXPECTED_COLUMNS}")
            print("Headers Verified (after auto-fix).")
    except Exception as error:
        print(f"Failed to verify/fix headers: {type(error).__name__}: {error}")


Current headers found in row 1: []
Header mismatch detected. Fixing automatically...
Headers rewritten to: ['lead_id', 'company_name', 'contact_name', 'email', 'phone', 'source', 'status', 'assigned_to', 'priority', 'country', 'industry', 'last_contact_date', 'notes']
Headers Verified (after auto-fix).


## Test google sheet 

In [6]:
TEST_LEAD_ID = "TEST_ROW_999"

if worksheet is None:
    print("Skipping: no worksheet connected (see Cell 4).")
else:
    try:
        test_row_values = [
            TEST_LEAD_ID, "Test Company", "Test Contact", "test@example.com",
            "+00-000-0000000", "Test", "Open", "Test User", "Low",
            "Testland", "Testing", str(datetime.now().date()), "This is a connectivity test row.",
        ]
        worksheet.append_row(test_row_values)
        print("Appended a test row. Reading the sheet back now...")

        records_after = worksheet.get_all_records()
        found = any(str(r.get("lead_id", "")) == TEST_LEAD_ID for r in records_after)

        if found:
            print("Row successfully inserted.")
            print(f"Sheet now has {len(records_after)} data row(s).")
        else:
            print("Write appeared to succeed but the test row was NOT found on read-back.")
            print("This would indicate you are reading a different worksheet/tab than you wrote to.")
    except APIError as error:
        print(f"Google Sheet Write Failed (API error): {error}")
        print("Fix: check that the service account has Editor (not just Viewer) access on this sheet.")
    except Exception as error:
        print(f"Google Sheet Write Failed: {type(error).__name__}: {error}")


Appended a test row. Reading the sheet back now...
Row successfully inserted.
Sheet now has 1 data row(s).


## Test row deletion

In [7]:
if worksheet is None:
    print("Skipping: no worksheet connected (see Cell 4).")
else:
    try:
        records = worksheet.get_all_records()
        row_number_to_delete = None
        for idx, record in enumerate(records, start=2):  # row 1 is the header, data starts at row 2
            if str(record.get("lead_id", "")) == TEST_LEAD_ID:
                row_number_to_delete = idx
                break

        if row_number_to_delete is None:
            print(f"Could not find a row with lead_id '{TEST_LEAD_ID}' — nothing to delete (Cell 6 may not have run).")
        else:
            worksheet.delete_rows(row_number_to_delete)
            print(f"Deleted row {row_number_to_delete}.")

            records_after = worksheet.get_all_records()
            still_present = any(str(r.get("lead_id", "")) == TEST_LEAD_ID for r in records_after)
            if not still_present:
                print("Test Row Deleted")
                print(f"Sheet now has {len(records_after)} data row(s).")
            else:
                print("Deletion did not take effect — the test row is still present on read-back.")
    except Exception as error:
        print(f"Failed to delete test row: {type(error).__name__}: {error}")


Deleted row 2.
Test Row Deleted
Sheet now has 0 data row(s).


## Class LeadManagementSystem 

In [8]:
class LeadManagementSystem:
    """
    A reusable Object-Oriented wrapper around a real gspread Worksheet
    that manages lead records for a sales/CRM-style workflow.

    The sheet must have exactly these columns, in this order:
        lead_id, company_name, contact_name, email, phone, source, status,
        assigned_to, priority, country, industry, last_contact_date, notes
    """

    COLUMNS = EXPECTED_COLUMNS

    def __init__(self, worksheet):
        """
        Args:
            worksheet: A connected gspread Worksheet object (see Cell 4).
        """
        self.worksheet = worksheet

    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------

    def _validate_lead_data(self, lead_data):
        """Validate a single lead dict against COLUMNS; fill missing keys with ''."""
        if not isinstance(lead_data, dict):
            raise ValueError("lead_data must be a dictionary.")
        unknown_keys = set(lead_data.keys()) - set(self.COLUMNS)
        if unknown_keys:
            raise ValueError(f"Unknown column(s) in lead_data: {sorted(unknown_keys)}")
        cleaned = {}
        for col in self.COLUMNS:
            value = lead_data.get(col, "")
            cleaned[col] = "" if value is None else str(value)
        return cleaned

    def _row_matches_filters(self, row, filters):
        """Case-insensitive match of a row dict against column=value filters."""
        for key, value in filters.items():
            if str(row.get(key, "")).strip().lower() != str(value).strip().lower():
                return False
        return True

    def _rewrite_sheet(self, rows):
        """
        Overwrite all rows in the worksheet with the given list of row dicts.

        IMPORTANT: worksheet.clear() wipes the header row too on a real Google
        Sheet, so we always re-insert EXPECTED_COLUMNS as row 1 immediately
        after clearing, before appending any data rows back.
        """
        self.worksheet.clear()
        self.worksheet.insert_row(self.COLUMNS, index=1)
        if rows:
            row_values = [[row.get(col, "") for col in self.COLUMNS] for row in rows]
            self.worksheet.append_rows(row_values)

    # ------------------------------------------------------------------
    # 1. add_row
    # ------------------------------------------------------------------

    def add_row(self, lead_data):
        """Add a single lead. Returns {"success", "message", "data"}."""
        try:
            cleaned = self._validate_lead_data(lead_data)
            if not cleaned["lead_id"]:
                return {"success": False, "message": "lead_id is required.", "data": None}
            if self.row_exists({"lead_id": cleaned["lead_id"]}):
                return {"success": False, "message": f"A lead with lead_id '{cleaned['lead_id']}' already exists.", "data": None}
            self.worksheet.append_row([cleaned[col] for col in self.COLUMNS])
            return {"success": True, "message": "Lead added successfully.", "data": cleaned}
        except Exception as error:
            return {"success": False, "message": f"Failed to add lead: {error}", "data": None}

    # ------------------------------------------------------------------
    # 2. get_all_rows
    # ------------------------------------------------------------------

    def get_all_rows(self):
        """Return every lead currently in the sheet as a list of dicts."""
        try:
            return self.worksheet.get_all_records()
        except Exception as error:
            print(f"Failed to fetch rows: {error}")
            return []

    # ------------------------------------------------------------------
    # 3. get_row_by_lead_id
    # ------------------------------------------------------------------

    def get_row_by_lead_id(self, lead_id):
        """Fetch a single lead by lead_id, or None if not found."""
        try:
            if not lead_id:
                return None
            for row in self.get_all_rows():
                if str(row.get("lead_id", "")).strip().lower() == str(lead_id).strip().lower():
                    return row
            return None
        except Exception as error:
            print(f"Failed to fetch lead by id: {error}")
            return None

    # ------------------------------------------------------------------
    # 4. get_rows_single_filter
    # ------------------------------------------------------------------

    def get_rows_single_filter(self, column_name, value):
        """Search for leads where a single column matches a given value."""
        try:
            if column_name not in self.COLUMNS:
                print(f"'{column_name}' is not a valid column.")
                return []
            return [row for row in self.get_all_rows() if self._row_matches_filters(row, {column_name: value})]
        except Exception as error:
            print(f"Failed to filter rows: {error}")
            return []

    # ------------------------------------------------------------------
    # 5. get_rows_multiple_filters
    # ------------------------------------------------------------------

    def get_rows_multiple_filters(self, **filters):
        """Search for leads matching multiple column filters simultaneously."""
        try:
            invalid_columns = set(filters.keys()) - set(self.COLUMNS)
            if invalid_columns:
                print(f"Invalid filter column(s): {sorted(invalid_columns)}")
                return []
            return [row for row in self.get_all_rows() if self._row_matches_filters(row, filters)]
        except Exception as error:
            print(f"Failed to filter rows: {error}")
            return []

    # ------------------------------------------------------------------
    # 6. update_row
    # ------------------------------------------------------------------

    def update_row(self, filters, updates):
        """Update all leads matching filters with new field values."""
        try:
            if not filters:
                return {"success": False, "message": "At least one filter is required.", "updated_count": 0}
            invalid_columns = set(updates.keys()) - set(self.COLUMNS)
            if invalid_columns:
                return {"success": False, "message": f"Invalid update column(s): {sorted(invalid_columns)}", "updated_count": 0}

            all_rows = self.get_all_rows()
            updated_count = 0
            for row in all_rows:
                if self._row_matches_filters(row, filters):
                    row.update({k: str(v) for k, v in updates.items()})
                    updated_count += 1

            if updated_count > 0:
                self._rewrite_sheet(all_rows)
                return {"success": True, "message": f"Updated {updated_count} lead(s).", "updated_count": updated_count}
            return {"success": False, "message": "No matching leads found.", "updated_count": 0}
        except Exception as error:
            return {"success": False, "message": f"Failed to update rows: {error}", "updated_count": 0}

    # ------------------------------------------------------------------
    # 7. delete_row
    # ------------------------------------------------------------------

    def delete_row(self, filters):
        """Delete all leads matching filters."""
        try:
            if not filters:
                return {"success": False, "message": "At least one filter is required.", "deleted_count": 0}
            all_rows = self.get_all_rows()
            remaining_rows = [row for row in all_rows if not self._row_matches_filters(row, filters)]
            deleted_count = len(all_rows) - len(remaining_rows)
            if deleted_count > 0:
                self._rewrite_sheet(remaining_rows)
                return {"success": True, "message": f"Deleted {deleted_count} lead(s).", "deleted_count": deleted_count}
            return {"success": False, "message": "No matching leads found.", "deleted_count": 0}
        except Exception as error:
            return {"success": False, "message": f"Failed to delete rows: {error}", "deleted_count": 0}

    # ------------------------------------------------------------------
    # 8. row_exists
    # ------------------------------------------------------------------

    def row_exists(self, filters):
        """Return True if at least one lead matches filters."""
        try:
            if not filters:
                return False
            return any(self._row_matches_filters(row, filters) for row in self.get_all_rows())
        except Exception as error:
            print(f"Failed to check row existence: {error}")
            return False

    # ------------------------------------------------------------------
    # 9. count_rows
    # ------------------------------------------------------------------

    def count_rows(self, filters=None):
        """Count leads, optionally restricted to those matching filters."""
        try:
            all_rows = self.get_all_rows()
            if not filters:
                return len(all_rows)
            return sum(1 for row in all_rows if self._row_matches_filters(row, filters))
        except Exception as error:
            print(f"Failed to count rows: {error}")
            return 0

    # ------------------------------------------------------------------
    # 10. add_multiple_rows
    # ------------------------------------------------------------------

    def add_multiple_rows(self, leads_data):
        """Add multiple leads in one batch. Returns success/added/skipped info."""
        try:
            if not isinstance(leads_data, list) or not leads_data:
                return {"success": False, "message": "leads_data must be a non-empty list.", "added_count": 0, "skipped": []}

            existing_ids = {str(r.get("lead_id", "")).strip().lower() for r in self.get_all_rows()}
            rows_to_add, skipped = [], []

            for lead_data in leads_data:
                try:
                    cleaned = self._validate_lead_data(lead_data)
                except Exception as validation_error:
                    skipped.append(f"{lead_data}: {validation_error}")
                    continue
                lead_id_key = cleaned["lead_id"].strip().lower()
                if not lead_id_key:
                    skipped.append(f"{lead_data}: missing lead_id")
                    continue
                if lead_id_key in existing_ids:
                    skipped.append(f"{cleaned['lead_id']}: duplicate lead_id")
                    continue
                existing_ids.add(lead_id_key)
                rows_to_add.append([cleaned[col] for col in self.COLUMNS])

            if rows_to_add:
                self.worksheet.append_rows(rows_to_add)

            return {
                "success": len(rows_to_add) > 0,
                "message": f"Added {len(rows_to_add)} lead(s), skipped {len(skipped)}.",
                "added_count": len(rows_to_add),
                "skipped": skipped,
            }
        except Exception as error:
            return {"success": False, "message": f"Failed to add multiple rows: {error}", "added_count": 0, "skipped": []}

    # ------------------------------------------------------------------
    # 11. get_unique_values
    # ------------------------------------------------------------------

    def get_unique_values(self, column_name):
        """Return sorted unique, non-empty values present in a given column."""
        try:
            if column_name not in self.COLUMNS:
                print(f"'{column_name}' is not a valid column.")
                return []
            values = {str(row.get(column_name, "")).strip() for row in self.get_all_rows()}
            values.discard("")
            return sorted(values)
        except Exception as error:
            print(f"Failed to get unique values: {error}")
            return []

    # ------------------------------------------------------------------
    # 12. export_dataframe
    # ------------------------------------------------------------------

    def export_dataframe(self):
        """Export the entire sheet as a pandas DataFrame."""
        try:
            rows = self.get_all_rows()
            if not rows:
                return pd.DataFrame(columns=self.COLUMNS)
            df = pd.DataFrame(rows)
            return df.reindex(columns=self.COLUMNS)
        except Exception as error:
            print(f"Failed to export DataFrame: {error}")
            return pd.DataFrame(columns=self.COLUMNS)


if worksheet is None:
    print("Skipping: no worksheet connected (see Cell 4). LeadManagementSystem was not instantiated.")
else:
    lms = LeadManagementSystem(worksheet)
    print("LeadManagementSystem is ready, wired to your real Google Sheet worksheet.")


LeadManagementSystem is ready, wired to your real Google Sheet worksheet.


## Verification

In [9]:
def show_sheet_state(label):
    """Print the current live state of the Google Sheet, straight from the API."""
    if worksheet is None:
        print(f"[{label}] Skipping — no worksheet connected.")
        return
    rows = worksheet.get_all_records()
    print(f"[{label}] Google Sheet now has {len(rows)} data row(s).")
    for row in rows[-5:]:
        print("   ", row)


## Loading sample data

In [10]:
sample_leads = [
    {"lead_id": "L001", "company_name": "Nexora Tech", "contact_name": "Ahmed Raza", "email": "ahmed.raza@nexoratech.com", "phone": "+92-300-1234567", "source": "Website", "status": "Open", "assigned_to": "Sara Khan", "priority": "High", "country": "Pakistan", "industry": "Software", "last_contact_date": "2026-07-01", "notes": "Interested in enterprise plan."},
    {"lead_id": "L002", "company_name": "GreenFields Agro", "contact_name": "Emily Chen", "email": "emily.chen@greenfields.com", "phone": "+1-415-5551234", "source": "Referral", "status": "Contacted", "assigned_to": "Bilal Ahmed", "priority": "Medium", "country": "USA", "industry": "Agriculture", "last_contact_date": "2026-06-28", "notes": "Requested a product demo."},
    {"lead_id": "L003", "company_name": "Bluewave Logistics", "contact_name": "Omar Farouk", "email": "omar.farouk@bluewave.ae", "phone": "+971-50-1234567", "source": "LinkedIn", "status": "Open", "assigned_to": "Sara Khan", "priority": "Low", "country": "UAE", "industry": "Logistics", "last_contact_date": "2026-06-20", "notes": "Still evaluating vendors."},
    {"lead_id": "L004", "company_name": "Sunrise Retail Group", "contact_name": "Maria Lopez", "email": "maria.lopez@sunriseretail.com", "phone": "+52-55-12345678", "source": "Trade Show", "status": "Won", "assigned_to": "Hamza Tariq", "priority": "High", "country": "Mexico", "industry": "Retail", "last_contact_date": "2026-05-15", "notes": "Signed annual contract."},
    {"lead_id": "L005", "company_name": "Zenith Financial Services", "contact_name": "David Miller", "email": "david.miller@zenithfs.com", "phone": "+44-20-79460123", "source": "Cold Call", "status": "Lost", "assigned_to": "Bilal Ahmed", "priority": "Medium", "country": "UK", "industry": "Finance", "last_contact_date": "2026-04-30", "notes": "Went with a competitor."},
    {"lead_id": "L006", "company_name": "Orion Healthcare", "contact_name": "Ayesha Malik", "email": "ayesha.malik@orionhealth.pk", "phone": "+92-321-9876543", "source": "Website", "status": "Open", "assigned_to": "Hamza Tariq", "priority": "High", "country": "Pakistan", "industry": "Healthcare", "last_contact_date": "2026-07-10", "notes": "Needs HIPAA compliance details."},
    {"lead_id": "L007", "company_name": "Falcon Steel Works", "contact_name": "John Smith", "email": "john.smith@falconsteel.com", "phone": "+1-312-5559876", "source": "Referral", "status": "Contacted", "assigned_to": "Sara Khan", "priority": "Low", "country": "USA", "industry": "Manufacturing", "last_contact_date": "2026-06-12", "notes": "Budget approval pending."},
    {"lead_id": "L008", "company_name": "Al-Noor Education Trust", "contact_name": "Fatima Sheikh", "email": "fatima.sheikh@alnoor.edu.pk", "phone": "+92-333-1122334", "source": "Email Campaign", "status": "Open", "assigned_to": "Bilal Ahmed", "priority": "Medium", "country": "Pakistan", "industry": "Education", "last_contact_date": "2026-07-05", "notes": "Wants a pilot for 3 campuses."},
    {"lead_id": "L009", "company_name": "Delta Energy Corp", "contact_name": "Lucas Fischer", "email": "lucas.fischer@deltaenergy.de", "phone": "+49-30-12345678", "source": "Trade Show", "status": "Contacted", "assigned_to": "Hamza Tariq", "priority": "High", "country": "Germany", "industry": "Energy", "last_contact_date": "2026-06-25", "notes": "Following up after conference."},
    {"lead_id": "L010", "company_name": "Coral Bay Hospitality", "contact_name": "Priya Nair", "email": "priya.nair@coralbay.in", "phone": "+91-98-76543210", "source": "Website", "status": "Won", "assigned_to": "Sara Khan", "priority": "Medium", "country": "India", "industry": "Hospitality", "last_contact_date": "2026-05-30", "notes": "Rolled out to 5 hotel branches."},
    {"lead_id": "L011", "company_name": "Skyline Real Estate", "contact_name": "Chris Anderson", "email": "chris.anderson@skylinere.com", "phone": "+1-646-5552211", "source": "LinkedIn", "status": "Open", "assigned_to": "Bilal Ahmed", "priority": "Low", "country": "USA", "industry": "Real Estate", "last_contact_date": "2026-06-18", "notes": "Comparing CRM options."},
    {"lead_id": "L012", "company_name": "Crescent Textiles", "contact_name": "Bushra Iqbal", "email": "bushra.iqbal@crescenttex.pk", "phone": "+92-42-35678901", "source": "Referral", "status": "Contacted", "assigned_to": "Hamza Tariq", "priority": "Medium", "country": "Pakistan", "industry": "Textiles", "last_contact_date": "2026-07-02", "notes": "Needs export-order tracking."},
    {"lead_id": "L013", "company_name": "Pinnacle Media Group", "contact_name": "Sophie Turner", "email": "sophie.turner@pinnaclemedia.co.uk", "phone": "+44-161-4960123", "source": "Cold Call", "status": "Lost", "assigned_to": "Sara Khan", "priority": "Low", "country": "UK", "industry": "Media", "last_contact_date": "2026-04-10", "notes": "Project postponed indefinitely."},
    {"lead_id": "L014", "company_name": "TerraFarm Solutions", "contact_name": "Carlos Mendes", "email": "carlos.mendes@terrafarm.br", "phone": "+55-11-98765432", "source": "Email Campaign", "status": "Open", "assigned_to": "Bilal Ahmed", "priority": "High", "country": "Brazil", "industry": "Agriculture", "last_contact_date": "2026-07-08", "notes": "Wants integration with existing ERP."},
    {"lead_id": "L015", "company_name": "Quantum Data Systems", "contact_name": "Hassan Abbasi", "email": "hassan.abbasi@quantumds.pk", "phone": "+92-51-8887766", "source": "Website", "status": "Contacted", "assigned_to": "Hamza Tariq", "priority": "High", "country": "Pakistan", "industry": "Software", "last_contact_date": "2026-07-15", "notes": "Requested a security audit report."},
]

if worksheet is None:
    print("Skipping: no worksheet connected (see Cell 4).")
else:
    batch_result = lms.add_multiple_rows(sample_leads)
    print(batch_result["message"])
    if batch_result["skipped"]:
        print("Skipped:", batch_result["skipped"])
    show_sheet_state("After loading sample data")


Added 15 lead(s), skipped 0.
[After loading sample data] Google Sheet now has 15 data row(s).
    {'lead_id': 'L011', 'company_name': 'Skyline Real Estate', 'contact_name': 'Chris Anderson', 'email': 'chris.anderson@skylinere.com', 'phone': '+1-646-5552211', 'source': 'LinkedIn', 'status': 'Open', 'assigned_to': 'Bilal Ahmed', 'priority': 'Low', 'country': 'USA', 'industry': 'Real Estate', 'last_contact_date': '2026-06-18', 'notes': 'Comparing CRM options.'}
    {'lead_id': 'L012', 'company_name': 'Crescent Textiles', 'contact_name': 'Bushra Iqbal', 'email': 'bushra.iqbal@crescenttex.pk', 'phone': '+92-42-35678901', 'source': 'Referral', 'status': 'Contacted', 'assigned_to': 'Hamza Tariq', 'priority': 'Medium', 'country': 'Pakistan', 'industry': 'Textiles', 'last_contact_date': '2026-07-02', 'notes': 'Needs export-order tracking.'}
    {'lead_id': 'L013', 'company_name': 'Pinnacle Media Group', 'contact_name': 'Sophie Turner', 'email': 'sophie.turner@pinnaclemedia.co.uk', 'phone': '+44

## Adding row

In [11]:
if worksheet is None:
    print("Skipping: no worksheet connected (see Cell 4).")
else:
    new_lead = {
        "lead_id": "L016", "company_name": "Aurora Cybersecurity", "contact_name": "Zainab Hussain",
        "email": "zainab.hussain@auroracyber.pk", "phone": "+92-300-5551122", "source": "LinkedIn",
        "status": "Open", "assigned_to": "Sara Khan", "priority": "High", "country": "Pakistan",
        "industry": "Cybersecurity", "last_contact_date": "2026-07-20", "notes": "Referred by an existing client.",
    }
    result = lms.add_row(new_lead)
    print(result)
    show_sheet_state("After add_row (L016)")


{'success': True, 'message': 'Lead added successfully.', 'data': {'lead_id': 'L016', 'company_name': 'Aurora Cybersecurity', 'contact_name': 'Zainab Hussain', 'email': 'zainab.hussain@auroracyber.pk', 'phone': '+92-300-5551122', 'source': 'LinkedIn', 'status': 'Open', 'assigned_to': 'Sara Khan', 'priority': 'High', 'country': 'Pakistan', 'industry': 'Cybersecurity', 'last_contact_date': '2026-07-20', 'notes': 'Referred by an existing client.'}}
[After add_row (L016)] Google Sheet now has 16 data row(s).
    {'lead_id': 'L012', 'company_name': 'Crescent Textiles', 'contact_name': 'Bushra Iqbal', 'email': 'bushra.iqbal@crescenttex.pk', 'phone': '+92-42-35678901', 'source': 'Referral', 'status': 'Contacted', 'assigned_to': 'Hamza Tariq', 'priority': 'Medium', 'country': 'Pakistan', 'industry': 'Textiles', 'last_contact_date': '2026-07-02', 'notes': 'Needs export-order tracking.'}
    {'lead_id': 'L013', 'company_name': 'Pinnacle Media Group', 'contact_name': 'Sophie Turner', 'email': 'sop

## Fetching leads  directly from the sheet

In [12]:
if worksheet is None:
    print("Skipping: no worksheet connected (see Cell 4).")
else:
    all_leads = lms.get_all_rows()
    print(f"Total leads: {len(all_leads)}")
    for row in all_leads[:3]:
        print(row)


Total leads: 16
{'lead_id': 'L001', 'company_name': 'Nexora Tech', 'contact_name': 'Ahmed Raza', 'email': 'ahmed.raza@nexoratech.com', 'phone': '+92-300-1234567', 'source': 'Website', 'status': 'Open', 'assigned_to': 'Sara Khan', 'priority': 'High', 'country': 'Pakistan', 'industry': 'Software', 'last_contact_date': '2026-07-01', 'notes': 'Interested in enterprise plan.'}
{'lead_id': 'L002', 'company_name': 'GreenFields Agro', 'contact_name': 'Emily Chen', 'email': 'emily.chen@greenfields.com', 'phone': '+1-415-5551234', 'source': 'Referral', 'status': 'Contacted', 'assigned_to': 'Bilal Ahmed', 'priority': 'Medium', 'country': 'USA', 'industry': 'Agriculture', 'last_contact_date': '2026-06-28', 'notes': 'Requested a product demo.'}
{'lead_id': 'L003', 'company_name': 'Bluewave Logistics', 'contact_name': 'Omar Farouk', 'email': 'omar.farouk@bluewave.ae', 'phone': '+971-50-1234567', 'source': 'LinkedIn', 'status': 'Open', 'assigned_to': 'Sara Khan', 'priority': 'Low', 'country': 'UAE', 

## Lead ID

In [13]:
if worksheet is None:
    print("Skipping: no worksheet connected (see Cell 4).")
else:
    print(lms.get_row_by_lead_id("L006"))


{'lead_id': 'L006', 'company_name': 'Orion Healthcare', 'contact_name': 'Ayesha Malik', 'email': 'ayesha.malik@orionhealth.pk', 'phone': '+92-321-9876543', 'source': 'Website', 'status': 'Open', 'assigned_to': 'Hamza Tariq', 'priority': 'High', 'country': 'Pakistan', 'industry': 'Healthcare', 'last_contact_date': '2026-07-10', 'notes': 'Needs HIPAA compliance details.'}


## Single filter

In [14]:
if worksheet is None:
    print("Skipping: no worksheet connected (see Cell 4).")
else:
    open_leads = lms.get_rows_single_filter("status", "Open")
    print(f"Open leads: {len(open_leads)}")


Open leads: 7


## Multiple filters

In [15]:
if worksheet is None:
    print("Skipping: no worksheet connected (see Cell 4).")
else:
    filtered_leads = lms.get_rows_multiple_filters(status="Open", country="Pakistan", priority="High")
    print(filtered_leads)


[{'lead_id': 'L001', 'company_name': 'Nexora Tech', 'contact_name': 'Ahmed Raza', 'email': 'ahmed.raza@nexoratech.com', 'phone': '+92-300-1234567', 'source': 'Website', 'status': 'Open', 'assigned_to': 'Sara Khan', 'priority': 'High', 'country': 'Pakistan', 'industry': 'Software', 'last_contact_date': '2026-07-01', 'notes': 'Interested in enterprise plan.'}, {'lead_id': 'L006', 'company_name': 'Orion Healthcare', 'contact_name': 'Ayesha Malik', 'email': 'ayesha.malik@orionhealth.pk', 'phone': '+92-321-9876543', 'source': 'Website', 'status': 'Open', 'assigned_to': 'Hamza Tariq', 'priority': 'High', 'country': 'Pakistan', 'industry': 'Healthcare', 'last_contact_date': '2026-07-10', 'notes': 'Needs HIPAA compliance details.'}, {'lead_id': 'L016', 'company_name': 'Aurora Cybersecurity', 'contact_name': 'Zainab Hussain', 'email': 'zainab.hussain@auroracyber.pk', 'phone': '+92-300-5551122', 'source': 'LinkedIn', 'status': 'Open', 'assigned_to': 'Sara Khan', 'priority': 'High', 'country': 'P

## Update row

In [16]:
if worksheet is None:
    print("Skipping: no worksheet connected (see Cell 4).")
else:
    update_result = lms.update_row(
        filters={"lead_id": "L003"},
        updates={"status": "Contacted", "notes": "Follow-up call scheduled for next week."},
    )
    print(update_result)
    show_sheet_state("After update_row (L003)")


{'success': True, 'message': 'Updated 1 lead(s).', 'updated_count': 1}
[After update_row (L003)] Google Sheet now has 16 data row(s).
    {'lead_id': 'L012', 'company_name': 'Crescent Textiles', 'contact_name': 'Bushra Iqbal', 'email': 'bushra.iqbal@crescenttex.pk', 'phone': '+92-42-35678901', 'source': 'Referral', 'status': 'Contacted', 'assigned_to': 'Hamza Tariq', 'priority': 'Medium', 'country': 'Pakistan', 'industry': 'Textiles', 'last_contact_date': '2026-07-02', 'notes': 'Needs export-order tracking.'}
    {'lead_id': 'L013', 'company_name': 'Pinnacle Media Group', 'contact_name': 'Sophie Turner', 'email': 'sophie.turner@pinnaclemedia.co.uk', 'phone': '+44-161-4960123', 'source': 'Cold Call', 'status': 'Lost', 'assigned_to': 'Sara Khan', 'priority': 'Low', 'country': 'UK', 'industry': 'Media', 'last_contact_date': '2026-04-10', 'notes': 'Project postponed indefinitely.'}
    {'lead_id': 'L014', 'company_name': 'TerraFarm Solutions', 'contact_name': 'Carlos Mendes', 'email': 'car

## Delete row

In [17]:
if worksheet is None:
    print("Skipping: no worksheet connected (see Cell 4).")
else:
    delete_result = lms.delete_row(filters={"lead_id": "L013"})
    print(delete_result)
    show_sheet_state("After delete_row (L013)")


{'success': True, 'message': 'Deleted 1 lead(s).', 'deleted_count': 1}
[After delete_row (L013)] Google Sheet now has 15 data row(s).
    {'lead_id': 'L011', 'company_name': 'Skyline Real Estate', 'contact_name': 'Chris Anderson', 'email': 'chris.anderson@skylinere.com', 'phone': '+1-646-5552211', 'source': 'LinkedIn', 'status': 'Open', 'assigned_to': 'Bilal Ahmed', 'priority': 'Low', 'country': 'USA', 'industry': 'Real Estate', 'last_contact_date': '2026-06-18', 'notes': 'Comparing CRM options.'}
    {'lead_id': 'L012', 'company_name': 'Crescent Textiles', 'contact_name': 'Bushra Iqbal', 'email': 'bushra.iqbal@crescenttex.pk', 'phone': '+92-42-35678901', 'source': 'Referral', 'status': 'Contacted', 'assigned_to': 'Hamza Tariq', 'priority': 'Medium', 'country': 'Pakistan', 'industry': 'Textiles', 'last_contact_date': '2026-07-02', 'notes': 'Needs export-order tracking.'}
    {'lead_id': 'L014', 'company_name': 'TerraFarm Solutions', 'contact_name': 'Carlos Mendes', 'email': 'carlos.men

## Exists

In [18]:
if worksheet is None:
    print("Skipping: no worksheet connected (see Cell 4).")
else:
    print("L001 exists:", lms.row_exists({"lead_id": "L001"}))
    print("L999 exists:", lms.row_exists({"lead_id": "L999"}))


L001 exists: True
L999 exists: False


## Count rows

In [19]:
if worksheet is None:
    print("Skipping: no worksheet connected (see Cell 4).")
else:
    print("Total leads:", lms.count_rows())
    print("High priority leads:", lms.count_rows(filters={"priority": "High"}))


Total leads: 15
High priority leads: 7


## Add multiple rows

In [20]:
if worksheet is None:
    print("Skipping: no worksheet connected (see Cell 4).")
else:
    more_leads = [
        {"lead_id": "L017", "company_name": "Ivory Consulting", "contact_name": "Nadia Farooq", "email": "nadia.farooq@ivoryconsulting.com", "phone": "+92-301-2223344", "source": "Referral", "status": "Open", "assigned_to": "Hamza Tariq", "priority": "Medium", "country": "Pakistan", "industry": "Consulting", "last_contact_date": "2026-07-18", "notes": "Needs a proposal by month end."},
        {"lead_id": "L018", "company_name": "Meridian Shipping", "contact_name": "Robert Hayes", "email": "robert.hayes@meridianshipping.com", "phone": "+1-713-5559988", "source": "Trade Show", "status": "Contacted", "assigned_to": "Bilal Ahmed", "priority": "Low", "country": "USA", "industry": "Logistics", "last_contact_date": "2026-07-12", "notes": "Comparing fleet-tracking vendors."},
    ]
    add_multiple_result = lms.add_multiple_rows(more_leads)
    print(add_multiple_result)
    show_sheet_state("After add_multiple_rows (L017, L018)")


{'success': True, 'message': 'Added 2 lead(s), skipped 0.', 'added_count': 2, 'skipped': []}
[After add_multiple_rows (L017, L018)] Google Sheet now has 17 data row(s).
    {'lead_id': 'L014', 'company_name': 'TerraFarm Solutions', 'contact_name': 'Carlos Mendes', 'email': 'carlos.mendes@terrafarm.br', 'phone': '+55-11-98765432', 'source': 'Email Campaign', 'status': 'Open', 'assigned_to': 'Bilal Ahmed', 'priority': 'High', 'country': 'Brazil', 'industry': 'Agriculture', 'last_contact_date': '2026-07-08', 'notes': 'Wants integration with existing ERP.'}
    {'lead_id': 'L015', 'company_name': 'Quantum Data Systems', 'contact_name': 'Hassan Abbasi', 'email': 'hassan.abbasi@quantumds.pk', 'phone': '+92-51-8887766', 'source': 'Website', 'status': 'Contacted', 'assigned_to': 'Hamza Tariq', 'priority': 'High', 'country': 'Pakistan', 'industry': 'Software', 'last_contact_date': '2026-07-15', 'notes': 'Requested a security audit report.'}
    {'lead_id': 'L016', 'company_name': 'Aurora Cybers

## Unique values

In [21]:
if worksheet is None:
    print("Skipping: no worksheet connected (see Cell 4).")
else:
    print("Unique countries:", lms.get_unique_values("country"))
    print("Unique industries:", lms.get_unique_values("industry"))


Unique countries: ['Brazil', 'Germany', 'India', 'Mexico', 'Pakistan', 'UAE', 'UK', 'USA']
Unique industries: ['Agriculture', 'Consulting', 'Cybersecurity', 'Education', 'Energy', 'Finance', 'Healthcare', 'Hospitality', 'Logistics', 'Manufacturing', 'Real Estate', 'Retail', 'Software', 'Textiles']


## Export dataframe

In [22]:
if worksheet is None:
    print("Skipping: no worksheet connected (see Cell 4).")
else:
    df = lms.export_dataframe()
    print(f"DataFrame shape: {df.shape}")
    display(df.head(10))


DataFrame shape: (17, 13)


,lead_id,company_name,contact_name,email,phone,source,status,assigned_to,priority,country,industry,last_contact_date,notes
0,L001,Nexora Tech,Ahmed Raza,ahmed.raza@nexoratech.com,+92-300-1234567,Website,Open,Sara Khan,High,Pakistan,Software,2026-07-01,Interested in enterprise plan.
1,L002,GreenFields Agro,Emily Chen,emily.chen@greenfields.com,+1-415-5551234,Referral,Contacted,Bilal Ahmed,Medium,USA,Agriculture,2026-06-28,Requested a product demo.
2,L003,Bluewave Logistics,Omar Farouk,omar.farouk@bluewave.ae,+971-50-1234567,LinkedIn,Contacted,Sara Khan,Low,UAE,Logistics,2026-06-20,Follow-up call scheduled for next week.
3,L004,Sunrise Retail Group,Maria Lopez,maria.lopez@sunriseretail.com,+52-55-12345678,Trade Show,Won,Hamza Tariq,High,Mexico,Retail,2026-05-15,Signed annual contract.
4,L005,Zenith Financial Services,David Miller,david.miller@zenithfs.com,+44-20-79460123,Cold Call,Lost,Bilal Ahmed,Medium,UK,Finance,2026-04-30,Went with a competitor.
5,L006,Orion Healthcare,Ayesha Malik,ayesha.malik@orionhealth.pk,+92-321-9876543,Website,Open,Hamza Tariq,High,Pakistan,Healthcare,2026-07-10,Needs HIPAA compliance details.
6,L007,Falcon Steel Works,John Smith,john.smith@falconsteel.com,+1-312-5559876,Referral,Contacted,Sara Khan,Low,USA,Manufacturing,2026-06-12,Budget approval pending.
7,L008,Al-Noor Education Trust,Fatima Sheikh,fatima.sheikh@alnoor.edu.pk,+92-333-1122334,Email Campaign,Open,Bilal Ahmed,Medium,Pakistan,Education,2026-07-05,Wants a pilot for 3 campuses.
8,L009,Delta Energy Corp,Lucas Fischer,lucas.fischer@deltaenergy.de,+49-30-12345678,Trade Show,Contacted,Hamza Tariq,High,Germany,Energy,2026-06-25,Following up after conference.
9,L010,Coral Bay Hospitality,Priya Nair,priya.nair@coralbay.in,+91-98-76543210,Website,Won,Sara Khan,Medium,India,Hospitality,2026-05-30,Rolled out to 5 hotel branches.


## Menu

In [23]:
def print_menu():
    print("\n==============================")
    print("   Lead Management System")
    print("==============================")
    print("1.  Add Lead")
    print("2.  View All Leads")
    print("3.  Get Lead by ID")
    print("4.  Search Lead (Single Filter)")
    print("5.  Search Leads (Multiple Filters)")
    print("6.  Update Lead")
    print("7.  Delete Lead")
    print("8.  Check if Lead Exists")
    print("9.  Count Leads")
    print("10. Add Multiple Leads")
    print("11. Get Unique Values from a Column")
    print("12. Export to DataFrame")
    print("13. Exit")


def prompt_lead_data():
    lead_data = {}
    for column in LeadManagementSystem.COLUMNS:
        lead_data[column] = input(f"  {column}: ").strip()
    return lead_data


def prompt_filters():
    raw = input("  Enter filters as column=value pairs, separated by commas: ").strip()
    filters = {}
    if not raw:
        return filters
    for pair in raw.split(","):
        if "=" not in pair:
            continue
        key, value = pair.split("=", 1)
        key, value = key.strip(), value.strip()
        if key in LeadManagementSystem.COLUMNS:
            filters[key] = value
        else:
            print(f"  (ignoring unknown column \'{key}\')")
    return filters


def run_console_menu(lms):
    while True:
        print_menu()
        choice = input("Select an option (1-13): ").strip()
        try:
            if choice == "1":
                lead_data = prompt_lead_data()
                result = lms.add_row(lead_data)
                print(("SUCCESS: " if result["success"] else "ERROR: ") + result["message"])
                show_sheet_state("After menu option 1 (Add Lead)")

            elif choice == "2":
                rows = lms.get_all_rows()
                print(f"Total leads: {len(rows)}")
                for row in rows:
                    print(row)

            elif choice == "3":
                lead_id = input("  Enter lead_id: ").strip()
                row = lms.get_row_by_lead_id(lead_id)
                print(row if row else f"No lead found with lead_id \'{lead_id}\'.")

            elif choice == "4":
                column = input("  Column to filter on: ").strip()
                value = input("  Value to match: ").strip()
                rows = lms.get_rows_single_filter(column, value)
                print(f"Found {len(rows)} matching lead(s).")
                for row in rows:
                    print(row)

            elif choice == "5":
                filters = prompt_filters()
                rows = lms.get_rows_multiple_filters(**filters)
                print(f"Found {len(rows)} matching lead(s).")
                for row in rows:
                    print(row)

            elif choice == "6":
                print("Filters (which leads to update):")
                filters = prompt_filters()
                print("Updates (new values):")
                updates = prompt_filters()
                result = lms.update_row(filters, updates)
                print(("SUCCESS: " if result["success"] else "ERROR: ") + result["message"])
                show_sheet_state("After menu option 6 (Update Lead)")

            elif choice == "7":
                filters = prompt_filters()
                result = lms.delete_row(filters)
                print(("SUCCESS: " if result["success"] else "ERROR: ") + result["message"])
                show_sheet_state("After menu option 7 (Delete Lead)")

            elif choice == "8":
                filters = prompt_filters()
                print("A matching lead exists." if lms.row_exists(filters) else "No matching lead found.")

            elif choice == "9":
                use_filter = input("  Apply a filter? (y/n): ").strip().lower()
                filters = prompt_filters() if use_filter == "y" else None
                print(f"Lead count: {lms.count_rows(filters)}")

            elif choice == "10":
                try:
                    n = int(input("  How many leads do you want to add? ").strip())
                except ValueError:
                    print("ERROR: please enter a whole number.")
                    continue
                leads_data = [prompt_lead_data() for _ in range(n)]
                result = lms.add_multiple_rows(leads_data)
                print(result["message"])
                show_sheet_state("After menu option 10 (Add Multiple Leads)")

            elif choice == "11":
                column = input("  Column name: ").strip()
                print(f"Unique values in \'{column}\':", lms.get_unique_values(column))

            elif choice == "12":
                df = lms.export_dataframe()
                print(f"Exported DataFrame with shape {df.shape}.")
                print(df)

            elif choice == "13":
                print("Goodbye!")
                break

            else:
                print("Invalid choice. Please select a number between 1 and 13.")
        except Exception as error:
            print(f"Something went wrong, but the menu is still running: {error}")


if worksheet is None:
    print("Console menu defined, but no worksheet is connected — fix Cells 3/4 first.")
else:
    print("Console menu is defined. Call run_console_menu(lms) to use it interactively:")
    print("    run_console_menu(lms)")


Console menu is defined. Call run_console_menu(lms) to use it interactively:
    run_console_menu(lms)


## Verification checklist

In [24]:
checklist = []

checklist.append(("Authentication successful", gc is not None))
checklist.append(("Spreadsheet opened successfully", 'spreadsheet' in dir() and spreadsheet is not None))
checklist.append(("Worksheet opened successfully", worksheet is not None))

headers_ok = False
if worksheet is not None:
    try:
        headers_ok = worksheet.row_values(1) == EXPECTED_COLUMNS
    except Exception:
        headers_ok = False
checklist.append(("Headers verified", headers_ok))

lms_ready = 'lms' in dir() and worksheet is not None
checklist.append(("LeadManagementSystem instantiated", lms_ready))

crud_ok = False
filters_ok = False
export_ok = False
if lms_ready:
    try:
        crud_ok = lms.row_exists({"lead_id": "L001"}) and lms.count_rows() > 0
        filters_ok = isinstance(lms.get_rows_multiple_filters(country="Pakistan"), list)
        export_ok = not lms.export_dataframe().empty
    except Exception:
        pass

checklist.append(("All CRUD operations working", crud_ok))
checklist.append(("Multiple filters working", filters_ok))
checklist.append(("DataFrame export working", export_ok))
checklist.append(("Console menu working", "run_console_menu" in dir()))

print("=" * 50)
print("FINAL VERIFICATION CHECKLIST")
print("=" * 50)
for label, passed in checklist:
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {label}")

all_passed = all(p for _, p in checklist)
print("=" * 50)
print("ALL CHECKS PASSED — your notebook is now correctly writing to Google Sheets." if all_passed
      else "Some checks failed — scroll up to the first FAIL and re-read that cell's printed error/fix.")


FINAL VERIFICATION CHECKLIST
[PASS] Authentication successful
[PASS] Spreadsheet opened successfully
[PASS] Worksheet opened successfully
[PASS] Headers verified
[PASS] LeadManagementSystem instantiated
[PASS] All CRUD operations working
[PASS] Multiple filters working
[PASS] DataFrame export working
[PASS] Console menu working
ALL CHECKS PASSED — your notebook is now correctly writing to Google Sheets.


## Output

In [25]:
run_console_menu(lms)


   Lead Management System
1.  Add Lead
2.  View All Leads
3.  Get Lead by ID
4.  Search Lead (Single Filter)
5.  Search Leads (Multiple Filters)
6.  Update Lead
7.  Delete Lead
8.  Check if Lead Exists
9.  Count Leads
10. Add Multiple Leads
11. Get Unique Values from a Column
12. Export to DataFrame
13. Exit


Select an option (1-13):  1
  lead_id:  L019
  company_name:  Galvan Ai
  contact_name:  Zuha Irfan
  email:  zuhairfan08@gmail.com
  phone:  +92343-5902786
  source:  Website
  status:  Open
  assigned_to:  Hajra Sarwar
  priority:  High
  country:  Pakistan
  industry:  Software
  last_contact_date:  '2026-10-25
  notes:  Needs webhook intergration with shopify


SUCCESS: Lead added successfully.
[After menu option 1 (Add Lead)] Google Sheet now has 18 data row(s).
    {'lead_id': 'L015', 'company_name': 'Quantum Data Systems', 'contact_name': 'Hassan Abbasi', 'email': 'hassan.abbasi@quantumds.pk', 'phone': '+92-51-8887766', 'source': 'Website', 'status': 'Contacted', 'assigned_to': 'Hamza Tariq', 'priority': 'High', 'country': 'Pakistan', 'industry': 'Software', 'last_contact_date': '2026-07-15', 'notes': 'Requested a security audit report.'}
    {'lead_id': 'L016', 'company_name': 'Aurora Cybersecurity', 'contact_name': 'Zainab Hussain', 'email': 'zainab.hussain@auroracyber.pk', 'phone': '+92-300-5551122', 'source': 'LinkedIn', 'status': 'Open', 'assigned_to': 'Sara Khan', 'priority': 'High', 'country': 'Pakistan', 'industry': 'Cybersecurity', 'last_contact_date': '2026-07-20', 'notes': 'Referred by an existing client.'}
    {'lead_id': 'L017', 'company_name': 'Ivory Consulting', 'contact_name': 'Nadia Farooq', 'email': 'nadia.farooq@ivorycon

Select an option (1-13):  2


Total leads: 18
{'lead_id': 'L001', 'company_name': 'Nexora Tech', 'contact_name': 'Ahmed Raza', 'email': 'ahmed.raza@nexoratech.com', 'phone': '+92-300-1234567', 'source': 'Website', 'status': 'Open', 'assigned_to': 'Sara Khan', 'priority': 'High', 'country': 'Pakistan', 'industry': 'Software', 'last_contact_date': '2026-07-01', 'notes': 'Interested in enterprise plan.'}
{'lead_id': 'L002', 'company_name': 'GreenFields Agro', 'contact_name': 'Emily Chen', 'email': 'emily.chen@greenfields.com', 'phone': '+1-415-5551234', 'source': 'Referral', 'status': 'Contacted', 'assigned_to': 'Bilal Ahmed', 'priority': 'Medium', 'country': 'USA', 'industry': 'Agriculture', 'last_contact_date': '2026-06-28', 'notes': 'Requested a product demo.'}
{'lead_id': 'L003', 'company_name': 'Bluewave Logistics', 'contact_name': 'Omar Farouk', 'email': 'omar.farouk@bluewave.ae', 'phone': '+971-50-1234567', 'source': 'LinkedIn', 'status': 'Contacted', 'assigned_to': 'Sara Khan', 'priority': 'Low', 'country': 'U

...

Unique values in 'company_name': ['Al-Noor Education Trust', 'Aurora Cybersecurity', 'Bluewave Logistics', 'Coral Bay Hospitality', 'Crescent Textiles', 'Delta Energy Corp', 'Falcon Steel Works', 'Galvan Ai', 'Global Retail Hub', 'GreenFields Agro', 'Ivory Consulting', 'Meridian Shipping', 'Nexora Tech', 'Orion Healthcare', 'Quantum Data Systems', 'Skyline Real Estate', 'Sunrise Retail Group', 'TechNova Solutions', 'TerraFarm Solutions', 'Zenith Financial Services']

   Lead Management System
1.  Add Lead
2.  View All Leads
3.  Get Lead by ID
4.  Search Lead (Single Filter)
5.  Search Leads (Multiple Filters)
6.  Update Lead
7.  Delete Lead
8.  Check if Lead Exists
9.  Count Leads
10. Add Multiple Leads
11. Get Unique Values from a Column
12. Export to DataFrame
13. Exit


Select an option (1-13):  12


Exported DataFrame with shape (20, 13).
   lead_id               company_name    contact_name  \
0     L001                Nexora Tech      Ahmed Raza   
1     L002           GreenFields Agro      Emily Chen   
2     L003         Bluewave Logistics     Omar Farouk   
3     L004       Sunrise Retail Group     Maria Lopez   
4     L005  Zenith Financial Services    David Miller   
5     L006           Orion Healthcare    Ayesha Malik   
6     L007         Falcon Steel Works      John Smith   
7     L008    Al-Noor Education Trust   Fatima Sheikh   
8     L009          Delta Energy Corp   Lucas Fischer   
9     L010      Coral Bay Hospitality      Priya Nair   
10    L011        Skyline Real Estate  Chris Anderson   
11    L012          Crescent Textiles    Bushra Iqbal   
12    L014        TerraFarm Solutions   Carlos Mendes   
13    L015       Quantum Data Systems   Hassan Abbasi   
14    L016       Aurora Cybersecurity  Zainab Hussain   
15    L017           Ivory Consulting    Nadia F

Select an option (1-13):  13


Goodbye!
